# BƯỚC 6: HYBRID CBF + CF CHO ỨNG DỤNG (THIẾT KẾ & KIỂM CHỨNG)

- **Notebook 05** xây hệ gợi ý dựa trên nội dung (CBF), đang chạy trong Streamlit.
- **Notebook 06** sinh dữ liệu tương tác mô phỏng và cho thấy: Matrix Factorization (ALS) học được sở thích "ẩn" mà CBF không thấy, nhưng **không gợi ý được tin mới**; kết hợp hai bên (Hybrid) tốt hơn từng bên.

Notebook này thiết kế **cách đưa Hybrid vào ứng dụng** và kiểm chứng từng quyết định trên bộ dữ liệu mô phỏng của notebook 06 (đọc lại từ `data/simulated/`, không sinh lại):

| Câu hỏi khi đưa vào ứng dụng | Mục |
|---|---|
| Kết hợp CBF và CF bằng công thức nào để điểm vẫn ở thang 0-100% và dễ giải thích? | 2 |
| Người dùng mới bấm vài tin thì tính gợi ý thế nào mà **không phải huấn luyện lại** ALS? | 2 |
| Tỷ trọng CF ($\lambda$) bao nhiêu để không làm chìm tin mới? | 3 |
| Người dùng cần **bao nhiêu lượt tương tác** thì mới nên bật CF? | 4 |
| CF thực sự học được gì mà CBF không thấy? | 5 |

> Dữ liệu mô phỏng: kết luận chỉ đúng dưới giả định của notebook 06; đọc **xu hướng và thứ tự**, không đọc giá trị tuyệt đối.

## 1. Nạp dữ liệu (tin thật + tương tác mô phỏng của notebook 06)

In [1]:
import os, re, json, time
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.stats import rankdata
from sklearn.feature_extraction.text import TfidfVectorizer

pd.set_option('display.width', 160)
ROOT = os.path.abspath(os.path.join('..', '..'))
SIM_DIR = os.path.join(ROOT, 'data', 'simulated')
CLEAN_CSV = os.path.join(ROOT, 'data', 'notebook', 'content_base', 'jobs_clean.csv')
if not os.path.exists(os.path.join(SIM_DIR, 'config.json')):
    raise FileNotFoundError("Chưa có data/simulated/: hãy chạy notebook 06 trước.")

config = json.load(open(os.path.join(SIM_DIR, 'config.json'), encoding='utf-8'))
users = pd.read_csv(os.path.join(SIM_DIR, 'users.csv'))
users['skills'] = users['skills'].fillna('').map(lambda s: [x.strip() for x in s.split(',') if x.strip()])
job_meta = pd.read_csv(os.path.join(SIM_DIR, 'jobs_hidden.csv'))          # thứ tự tin = thứ tự cột của ma trận U
interactions = pd.read_csv(os.path.join(SIM_DIR, 'interactions_alpha0.5.csv'))
U_TRUE = np.load(os.path.join(SIM_DIR, 'true_utility_alpha0.5.npy')).astype(np.float32)

clean = pd.read_csv(CLEAN_CSV).set_index('job_id')
missing = set(job_meta['job_id']) - set(clean.index)
if missing:
    raise ValueError(f"{len(missing):,} tin của bộ mô phỏng không còn trong jobs_clean.csv: hãy chạy lại notebook 06.")
jobs = clean.loc[job_meta['job_id']].reset_index()
jobs['skills'] = jobs['required_skills'].fillna('').map(lambda s: [x.strip() for x in s.split(',') if x.strip()])
jobs = pd.concat([jobs, job_meta[['domain', 'open_day', 'close_day', 'company_type']].reset_index(drop=True)], axis=1)

job_idx = {j: i for i, j in enumerate(jobs['job_id'])}
interactions['job_idx'] = interactions['job_id'].map(job_idx)
N_USERS, N_ITEMS = len(users), len(jobs)
VAL_START, TEST_START, K = config['val_start_day'], config['test_start_day'], 10
ALS_PARAMS = config['als_params']
open_d, close_d = jobs['open_day'].values, jobs['close_day'].values

print(f"Dữ liệu mô phỏng ngày {config['snapshot']}: {N_USERS:,} người dùng × {N_ITEMS:,} tin, {len(interactions):,} sự kiện")
print("Phân bố sự kiện:", interactions['event'].value_counts().to_dict())
print("Tham số ALS chọn ở notebook 06:", ALS_PARAMS)

Dữ liệu mô phỏng ngày 2026-11-06: 3,000 người dùng × 3,550 tin, 24,971 sự kiện
Phân bố sự kiện: {'click': 17746, 'save': 4769, 'apply': 2456}
Tham số ALS chọn ở notebook 06: {'k': 8, 'reg': 0.1, 'alpha_conf': 1.0}


### 1.1. Các hàm dùng chung (giống notebook 06)
Chia tập theo thời gian (train trước ngày 45, test 10 ngày cuối; validation = ngày 38-44), các chỉ số Recall@10 / NDCG@10 / Oracle-P@10 / Recall@10 trên tin mới, ALS cho phản hồi ngầm và CBF rút gọn. Viết lại ngay trong notebook theo nguyên tắc notebook-first (không import từ notebook khác hay từ `src/`).

In [2]:
def build_split(ev, start=TEST_START, end=10_000):
    tr, te = ev[ev.day < start], ev[(ev.day >= start) & (ev.day < end)]
    R = tr.groupby(['user_id', 'job_idx'])['weight'].max().reset_index()
    R = sp.csr_matrix((R['weight'].values.astype(np.float32), (R['user_id'].values, R['job_idx'].values)), shape=(N_USERS, N_ITEMS))
    gains = te.assign(g=te['event'].map({'click': 1, 'save': 2, 'apply': 3})).groupby(['user_id', 'job_idx'])['g'].max()
    return {'R': R, 'gains': gains, 'eval_users': np.array(sorted(set(te.user_id) & set(tr.user_id))),
            'cand': (close_d >= start) & (open_d < end), 'warm_items': np.asarray((R > 0).sum(0)).ravel() > 0}

def evaluate(scores, split, U_true, rows=None):
    """Chỉ số trung bình; rows: chỉ tính trên một nhóm người dùng (vị trí trong eval_users)."""
    R, cand, warm = split['R'], split['cand'], split['warm_items']
    rec, ndcg, orac, cold_hit, cold_tot = [], [], [], 0, 0
    disc = 1 / np.log2(np.arange(2, K + 2))
    for row in (range(len(split['eval_users'])) if rows is None else rows):
        u = split['eval_users'][row]
        mask = cand.copy(); mask[R[u].indices] = False
        s = np.where(mask, scores[row], -np.inf)
        top = np.argpartition(-s, K)[:K]; top = top[np.argsort(-s[top])]
        pos = {int(j): v for j, v in split['gains'].loc[u].items() if mask[int(j)]}
        if pos:
            hit = [pos.get(int(j), 0) for j in top]
            rec.append(sum(h > 0 for h in hit) / min(K, len(pos)))
            ideal = sorted(pos.values(), reverse=True)[:K]
            ndcg.append(np.sum((2 ** np.array(hit) - 1) * disc) / np.sum((2 ** np.array(ideal) - 1) * disc[:len(ideal)]))
            cold = [j for j in pos if not warm[j]]
            cold_tot += len(cold); cold_hit += len(set(cold) & set(top.tolist()))
        ut = np.where(mask, U_true[u], -np.inf)
        orac.append(len(set(top.tolist()) & set(np.argpartition(-ut, K)[:K].tolist())) / K)
    return {'Recall@10': np.mean(rec), 'NDCG@10': np.mean(ndcg), 'Oracle-P@10': np.mean(orac),
            'Recall@10 tin mới': cold_hit / cold_tot if cold_tot else np.nan, 'Số người dùng': len(orac)}

def als_implicit(R, k=16, reg=1.0, alpha_conf=1.0, iters=12, seed=0):
    rng_ = np.random.default_rng(seed)
    X = rng_.normal(0, 0.01, (R.shape[0], k)); Y = rng_.normal(0, 0.01, (R.shape[1], k))
    Rt, I = R.T.tocsr(), reg * np.eye(k)
    def update(A, B, M):
        BtB = B.T @ B
        for r in range(M.shape[0]):
            s, e = M.indptr[r], M.indptr[r + 1]
            if s == e:
                A[r] = 0; continue
            idx, c = M.indices[s:e], 1 + alpha_conf * M.data[s:e]
            Bi = B[idx]
            A[r] = np.linalg.solve(BtB + (Bi.T * (c - 1)) @ Bi + I, Bi.T @ c)
    for _ in range(iters):
        update(X, Y, R); update(Y, X, Rt)
    return X, Y

def scores_als(split):
    X, Y = als_implicit(split['R'], **ALS_PARAMS)
    return X[split['eval_users']] @ Y.T

# CBF rút gọn (như notebook 06): TF-IDF (tiêu đề + kỹ năng) so với kỹ năng hồ sơ + kinh nghiệm + thành phố
tfidf = TfidfVectorizer(token_pattern=r"(?u)[\w+#.]+", ngram_range=(1, 2), min_df=2)
JOB_TEXT = tfidf.fit_transform(jobs['title'].fillna('') + ' ' + jobs['skills'].map(lambda s: ' '.join(s * 2)))
USER_TEXT = tfidf.transform(users['skills'].map(' '.join))
job_exp = jobs['experience_min'].fillna(0).values
ucity, jcity = users['city'].to_numpy(dtype=object)[:, None], jobs['city'].to_numpy(dtype=object)[None, :]
CBF_ALL = ((USER_TEXT @ JOB_TEXT.T).toarray()
           + 0.2 * np.clip(1 - np.maximum(job_exp[None, :] - users['exp'].values[:, None], 0) / 3, 0, 1)
           + 0.1 * np.where((ucity == 'Tất cả') | (ucity == jcity), 1.0, 0.0))
scores_cbf = lambda split: CBF_ALL[split['eval_users']]

def zrow(S, mask):
    S = np.where(mask, S, np.nan)
    return np.nan_to_num((S - np.nanmean(S, 1, keepdims=True)) / (np.nanstd(S, 1, keepdims=True) + 1e-9))

def scores_hybrid_z(split, als, w_cbf=config['w_cbf']):
    """Hybrid của notebook 06: z-score theo người dùng; tin mới chỉ dùng CBF."""
    zc, za = zrow(scores_cbf(split), split['cand']), zrow(als, split['cand'])
    return np.where(split['warm_items'][None, :], w_cbf * zc + (1 - w_cbf) * za, zc)

split = build_split(interactions)
val = build_split(interactions, VAL_START, TEST_START)
print(f"Test: {len(split['eval_users']):,} người dùng; validation: {len(val['eval_users']):,} người dùng; "
      f"{(~split['warm_items'] & split['cand']).sum() / split['cand'].sum():.0%} tin ứng viên ở giai đoạn test chưa có tương tác")

Test: 1,441 người dùng; validation: 1,086 người dùng; 43% tin ứng viên ở giai đoạn test chưa có tương tác


## 2. Hai công thức kết hợp ứng viên

**(A) Trộn tuyến tính trên thang 0-1** (dễ hiển thị dạng %):
$$\text{Điểm} = (1 - \lambda) \cdot \text{CBF}_{[0,1]} + \lambda \cdot \text{CF}_{[0,1]}$$
CBF đưa về $[0, 1]$ bằng min-max theo người dùng; CF là **thứ hạng phần trăm** của điểm ALS trong các tin đã có tương tác; tin mới nhận CF = 0,5 (trung lập). Có thêm biến thể CF min-max làm đối chứng.

**(B) Trộn z-score** (notebook 06):
$$\text{Điểm} = w \cdot z(\text{CBF}) + (1 - w) \cdot z(\text{ALS})$$
chuẩn hóa z-score theo từng người dùng; tin mới (ALS không biết) chỉ dùng $z(\text{CBF})$, cùng quy tắc "thiếu dữ liệu thì bỏ thành phần" của notebook 05.

**Người dùng mới: fold-in thay vì huấn luyện lại.** Giữ nguyên vector tin $Y$ đã học, chỉ giải **một** hệ $k \times k$ cho vector người dùng (đúng công thức cập nhật $x_u$ của ALS):
$$x_u = (Y^\top Y + Y^\top (C_u - I) Y + \lambda_{reg} I)^{-1} Y^\top C_u p_u$$
Để kiểm chứng, ALS được học **không có** các người dùng được đánh giá, rồi tính vector của họ bằng fold-in từ lịch sử train của chính họ.

In [3]:
def fold_in(Y, item_idx, weights, reg=None, alpha_conf=None):
    """Vector người dùng mới từ các tin đã tương tác, giữ nguyên vector tin Y (một bước cập nhật ALS)."""
    reg = ALS_PARAMS['reg'] if reg is None else reg
    alpha_conf = ALS_PARAMS['alpha_conf'] if alpha_conf is None else alpha_conf
    Yi, c = Y[item_idx], 1 + alpha_conf * np.asarray(weights, dtype=float)
    return np.linalg.solve(Y.T @ Y + (Yi.T * (c - 1)) @ Yi + reg * np.eye(Y.shape[1]), Yi.T @ c)

def minmax_rows(S, mask):
    S = np.where(mask, S, np.nan)
    lo, hi = np.nanmin(S, 1, keepdims=True), np.nanmax(S, 1, keepdims=True)
    return np.nan_to_num((S - lo) / np.where(hi > lo, hi - lo, 1.0))

def percentile_rows(S, mask, fill=0.5):
    """Thứ hạng phần trăm (0, 1] trên các cột mask; cột ngoài mask nhận giá trị trung lập `fill`."""
    out, idx = np.full(S.shape, fill, dtype=float), np.flatnonzero(mask)
    if len(idx):
        out[:, idx] = np.apply_along_axis(rankdata, 1, S[:, idx]) / len(idx)
    return out

def blend_linear(split, cbf, als, lam, cf_norm='percentile'):
    """(A) (1-λ)·CBF + λ·CF trên thang 0-1."""
    cand, warm = split['cand'], split['warm_items']
    c01 = minmax_rows(cbf, cand)
    if cf_norm == 'minmax':          # đối chứng: min-max, tin mới bỏ thành phần CF
        return np.where(warm[None, :], (1 - lam) * c01 + lam * minmax_rows(als, cand & warm), c01)
    return (1 - lam) * c01 + lam * percentile_rows(als, cand & warm)

def blend_z(split, cbf, als, w):
    """(B) w·z(CBF) + (1-w)·z(ALS); tin mới chỉ dùng z(CBF)."""
    zc, za = zrow(cbf, split['cand']), zrow(als, split['cand'] & split['warm_items'])
    return np.where(split['warm_items'][None, :], w * zc + (1 - w) * za, zc)

def als_scores_foldin(split):
    R = split['R'].tolil(copy=True)
    for u in split['eval_users']:
        R.rows[u], R.data[u] = [], []
    _, Y = als_implicit(R.tocsr(), **ALS_PARAMS)
    X = np.stack([fold_in(Y, split['R'][u].indices, split['R'][u].data) for u in split['eval_users']])
    return X @ Y.T

t0 = time.time()
cbf_val, als_val = scores_cbf(val), scores_als(val)
cbf_test, als_full = scores_cbf(split), scores_als(split)
als_fold = als_scores_foldin(split)
print(f"Đã tính điểm CBF, ALS đầy đủ và ALS fold-in ({time.time() - t0:.0f}s)")

Đã tính điểm CBF, ALS đầy đủ và ALS fold-in (2s)


## 3. Chọn công thức và tham số trên validation

**Thước đo để chọn: Oracle-P@10 (sở thích thật), không phải NDCG theo log.** Notebook 06 (mục 8, ý 3) đã cho thấy log hành vi **thiên vị** hệ thống đang hiển thị tin (ở đây là hệ dựa trên nội dung), nên chọn theo NDCG log sẽ ưu ái các công thức "giống CBF". Trong mô phỏng ta biết sở thích thật nên dùng được Oracle; với log thật thì phải dùng thước đo có hiệu chỉnh thiên lệch (vd. inverse propensity scoring).

**Ràng buộc:** Recall@10 trên **tin mới** phải còn ít nhất 50% của CBF. Tin tuyển dụng chỉ sống vài tuần, một công thức làm chìm tin mới sẽ lặp lại đúng điểm yếu của ALS.

Quy tắc: trong các cấu hình thỏa ràng buộc, chọn cấu hình có Oracle-P@10 cao nhất.

In [4]:
CFG = {}
for lam in [0.1, 0.2, 0.3, 0.4, 0.5]:
    CFG[f'(A) tuyến tính, λ = {lam}'] = ('linear', lam)
for w in [0.5, 0.6, 0.7, 0.8, 0.9]:
    CFG[f'(B) z-score, w_CBF = {w}'] = ('z', w)

def apply_cfg(cfg, split_, cbf, als):
    kind, p = cfg
    return blend_linear(split_, cbf, als, p) if kind == 'linear' else blend_z(split_, cbf, als, p)

cbf_val_cold = evaluate(cbf_val, val, U_TRUE)['Recall@10 tin mới']
grid = pd.DataFrame({name: evaluate(apply_cfg(cfg, val, cbf_val, als_val), val, U_TRUE) for name, cfg in CFG.items()}).T
grid = grid[['Oracle-P@10', 'NDCG@10', 'Recall@10 tin mới']]
grid['Giữ ≥ 50% tin mới của CBF'] = grid['Recall@10 tin mới'] >= 0.5 * cbf_val_cold
display(grid.round(4))
ok = grid[grid['Giữ ≥ 50% tin mới của CBF']]
BEST = ok['Oracle-P@10'].idxmax()
BEST_CFG = CFG[BEST]
print(f"CBF trên validation: Oracle-P@10 = {evaluate(cbf_val, val, U_TRUE)['Oracle-P@10']:.3f}, "
      f"Recall@10 tin mới = {cbf_val_cold:.3f}\n➔ Chọn: {BEST}")

best_lin = ok[ok.index.str.startswith('(A)')]['Oracle-P@10'].idxmax() if ok.index.str.startswith('(A)').any() else '(A) tuyến tính, λ = 0.2'
product = pd.DataFrame({
    'CBF': evaluate(cbf_test, split, U_TRUE),
    'ALS': evaluate(als_full, split, U_TRUE),
    f'Đối chứng: {best_lin}, CF min-max': evaluate(blend_linear(split, cbf_test, als_full, CFG[best_lin][1], 'minmax'), split, U_TRUE),
    f'{best_lin} (ALS đầy đủ)': evaluate(apply_cfg(CFG[best_lin], split, cbf_test, als_full), split, U_TRUE),
    f'{BEST} (ALS đầy đủ)': evaluate(apply_cfg(BEST_CFG, split, cbf_test, als_full), split, U_TRUE),
    f'{BEST} (fold-in) ★ ứng dụng': evaluate(apply_cfg(BEST_CFG, split, cbf_test, als_fold), split, U_TRUE),
}).T
print("Kết quả trên tập TEST:")
display(product.drop(columns='Số người dùng').round(3))

,Oracle-P@10,NDCG@10,Recall@10 tin mới,Giữ ≥ 50% tin mới của CBF
"(A) tuyến tính, λ = 0.1",0.1110,0.0547,0.0584,True
"(A) tuyến tính, λ = 0.2",0.1297,0.0667,0.0449,True
"(A) tuyến tính, λ = 0.3",0.1513,0.0750,0.0157,False
"(A) tuyến tính, λ = 0.4",0.1685,0.0836,0.0090,False
"(A) tuyến tính, λ = 0.5",0.1808,0.0868,0.0000,False
"(B) z-score, w_CBF = 0.5",0.1610,0.0544,0.0831,True
"(B) z-score, w_CBF = 0.6",0.1562,0.0559,0.1034,True
"(B) z-score, w_CBF = 0.7",0.1422,0.0542,0.1101,True
"(B) z-score, w_CBF = 0.8",0.1243,0.0509,0.1213,True
"(B) z-score, w_CBF = 0.9",0.1058,0.0473,0.1056,True


CBF trên validation: Oracle-P@10 = 0.094, Recall@10 tin mới = 0.090
➔ Chọn: (B) z-score, w_CBF = 0.5
Kết quả trên tập TEST:


,Recall@10,NDCG@10,Oracle-P@10,Recall@10 tin mới
CBF,0.083,0.054,0.095,0.053
ALS,0.059,0.037,0.174,0.000
"Đối chứng: (A) tuyến tính, λ = 0.2, CF min-max",0.067,0.041,0.092,0.089
"(A) tuyến tính, λ = 0.2 (ALS đầy đủ)",0.115,0.075,0.138,0.032
"(B) z-score, w_CBF = 0.5 (ALS đầy đủ)",0.073,0.048,0.181,0.058
"(B) z-score, w_CBF = 0.5 (fold-in) ★ ứng dụng",0.059,0.040,0.151,0.058


## 4. Cần bao nhiêu lượt tương tác thì nên bật CF?
Với người dùng chỉ bấm 1-2 tin, vector fold-in còn rất "nhiễu". Chia người dùng được đánh giá theo số tin đã tương tác ở train, so sánh Hybrid (fold-in) với CBF trong từng nhóm. Ngưỡng bật CF trong ứng dụng = nhóm nhỏ nhất mà Hybrid đã tốt hơn CBF theo **Oracle-P@10** (cùng thước đo dùng để chọn công thức ở mục 3; NDCG theo log chỉ để tham khảo).

In [5]:
n_train = np.diff(split['R'].indptr)[split['eval_users']]
BUCKETS = [(1, 2), (3, 5), (6, 10), (11, 10_000)]
hyb_fold = apply_cfg(BEST_CFG, split, cbf_test, als_fold)
rows = []
for lo, hi in BUCKETS:
    sel = np.flatnonzero((n_train >= lo) & (n_train <= hi))
    if len(sel) == 0:
        continue
    c, h = evaluate(cbf_test, split, U_TRUE, sel), evaluate(hyb_fold, split, U_TRUE, sel)
    rows.append({'Số tin đã tương tác': f"{lo}-{hi}" if hi < 10_000 else f"≥ {lo}", 'Số người dùng': len(sel),
                 'Oracle-P@10 CBF': c['Oracle-P@10'], 'Oracle-P@10 Hybrid': h['Oracle-P@10'],
                 'NDCG@10 CBF': c['NDCG@10'], 'NDCG@10 Hybrid': h['NDCG@10'], 'lo': lo})
by_n = pd.DataFrame(rows).set_index('Số tin đã tương tác')
display(by_n.drop(columns='lo').round(3))
better = by_n[by_n['Oracle-P@10 Hybrid'] > by_n['Oracle-P@10 CBF']]
MIN_USER_INTERACTIONS = int(better['lo'].min()) if len(better) else None
if MIN_USER_INTERACTIONS is None:
    raise ValueError('Hybrid không hơn CBF ở nhóm nào: không nên bật CF trong ứng dụng.')
print(f"Bật CF khi người dùng có ít nhất {MIN_USER_INTERACTIONS} tin đã tương tác")

,Số người dùng,Oracle-P@10 CBF,Oracle-P@10 Hybrid,NDCG@10 CBF,NDCG@10 Hybrid
Số tin đã tương tác,,,,,
1-2,227,0.083,0.145,0.044,0.037
3-5,351,0.102,0.151,0.039,0.030
6-10,430,0.097,0.153,0.065,0.045
≥ 11,433,0.092,0.152,0.062,0.045


Bật CF khi người dùng có ít nhất 1 tin đã tương tác


## 5. Minh họa: CF học được gì mà CBF không thấy?
Trong bộ mô phỏng, mỗi người dùng có **khẩu vị ẩn về loại công ty** (Startup / Product / Outsourcing / Tài chính - DN lớn) mà hồ sơ không ghi. Lấy một người dùng có khẩu vị rõ, so sánh loại công ty trong top-10 của CBF và của Hybrid với loại công ty họ đã bấm.

In [6]:
TYPES = ['Startup', 'Product', 'Outsourcing', 'Tài chính / DN lớn']
taste = np.stack(users['taste'].map(json.loads).values)
row = int(np.argmax([taste[u].max() * (n >= 5) for u, n in zip(split['eval_users'], n_train)]))
u = split['eval_users'][row]
mask = split['cand'].copy(); mask[split['R'][u].indices] = False

def top10(S):
    s = np.where(mask, S[row], -np.inf)
    return np.argsort(-s)[:K]

def type_share(idx):
    return pd.Series(jobs.loc[idx, 'company_type'].values).value_counts(normalize=True).reindex(TYPES, fill_value=0)

demo = pd.DataFrame({
    'Khẩu vị thật (ẩn)': pd.Series(taste[u], index=TYPES),
    'Tin đã bấm (train)': type_share(split['R'][u].indices),
    'Top-10 CBF': type_share(top10(cbf_test)),
    'Top-10 Hybrid': type_share(top10(hyb_fold)),
}).round(2)
print(f"Người dùng {u}: nhóm {users.loc[u, 'domain']}, kỹ năng {', '.join(users.loc[u, 'skills'][:6])}; {n_train[row]} tin đã tương tác")
display(demo)
print(f"Oracle-P@10 của người dùng này: CBF {evaluate(cbf_test, split, U_TRUE, [row])['Oracle-P@10']:.1f}, "
      f"Hybrid {evaluate(hyb_fold, split, U_TRUE, [row])['Oracle-P@10']:.1f}")

Người dùng 2788: nhóm Data/AI, kỹ năng Hadoop, Azure, ETL; 24 tin đã tương tác


,Khẩu vị thật (ẩn),Tin đã bấm (train),Top-10 CBF,Top-10 Hybrid
Startup,0.12,0.04,0.1,0.1
Product,0.01,0.17,0.4,0.1
Outsourcing,0.00,0.25,0.4,0.4
Tài chính / DN lớn,0.87,0.54,0.1,0.4


Oracle-P@10 của người dùng này: CBF 0.1, Hybrid 0.1


## 6. Kiểm tra tự động & thông số đưa vào ứng dụng
Ô dưới tự kiểm tra lại từng nhận định trên lần chạy hiện tại (chạy lại khi dữ liệu thay đổi), rồi in ra các thông số mà `src/recommender/hybrid.py` dùng.

In [7]:
def check(label, ok, detail):
    print(f"[{'ĐÚNG' if ok else 'KHÔNG CÒN ĐÚNG'}] {label}: {detail}")

O, P = product['Oracle-P@10'], product['NDCG@10']
app, full = f'{BEST} (fold-in) ★ ứng dụng', f'{BEST} (ALS đầy đủ)'
lin_full, lin_mm = f'{best_lin} (ALS đầy đủ)', f'Đối chứng: {best_lin}, CF min-max'
check("Hybrid (ứng dụng) hơn CBF thuần theo sở thích thật", O[app] > O['CBF'], f"Oracle-P@10 {O[app]:.3f} so với {O['CBF']:.3f}")
check("Fold-in (người dùng mới, không huấn luyện lại) vẫn hơn CBF thuần", O[app] > O['CBF'], f"Oracle-P@10 {O[app]:.3f} so với {O['CBF']:.3f}")
check("Hybrid giữ được tin mới (ALS thuần thì không)",
      product.loc[app, 'Recall@10 tin mới'] > 0 and product.loc['ALS', 'Recall@10 tin mới'] == 0,
      f"{product.loc[app, 'Recall@10 tin mới']:.3f} (CBF {product.loc['CBF', 'Recall@10 tin mới']:.3f}, ALS {product.loc['ALS', 'Recall@10 tin mới']:.3f})")
check("Chọn theo log (NDCG) và theo sở thích thật (Oracle) cho kết quả KHÁC nhau (log thiên lệch)",
      ok['NDCG@10'].idxmax() != ok['Oracle-P@10'].idxmax(),
      f"NDCG chọn '{ok['NDCG@10'].idxmax()}', Oracle chọn '{ok['Oracle-P@10'].idxmax()}'")
check("CF min-max kém hơn CF thứ hạng phần trăm (công thức A)", O[lin_mm] < O[lin_full], f"{O[lin_mm]:.3f} so với {O[lin_full]:.3f}")

print()
print("THAM KHẢO (không phải tiêu chí đạt/không đạt):")
print(f"  • Theo log, Hybrid {'cao' if P[app] > P['CBF'] else 'thấp'} hơn CBF: NDCG@10 {P[app]:.3f} so với {P['CBF']:.3f}. "
      "Log thiên vị hệ thống đã hiển thị tin (hệ dựa trên nội dung), xem notebook 06 mục 8.")
print(f"  • Fold-in đạt {O[app] / O[full]:.0%} Oracle-P@10 của huấn luyện lại đầy đủ ({O[app]:.3f} so với {O[full]:.3f}). "
      "Ở quy mô hiện tại huấn luyện lại ALS chỉ mất khoảng 1 giây, nên ứng dụng huấn luyện lại khi có tương tác mới; "
      "fold-in là phương án khi dữ liệu lớn.")

kind, p = BEST_CFG
print()
print("Thông số cho src/recommender/hybrid.py:")
print(f"  BLEND = {'zscore' if kind == 'z' else 'linear'}   {'W_CBF' if kind == 'z' else 'LAMBDA_CF'} = {p}   "
      f"MIN_USER_INTERACTIONS = {MIN_USER_INTERACTIONS}   ALS_PARAMS = {ALS_PARAMS}")

[ĐÚNG] Hybrid (ứng dụng) hơn CBF thuần theo sở thích thật: Oracle-P@10 0.151 so với 0.095
[ĐÚNG] Fold-in (người dùng mới, không huấn luyện lại) vẫn hơn CBF thuần: Oracle-P@10 0.151 so với 0.095
[ĐÚNG] Hybrid giữ được tin mới (ALS thuần thì không): 0.058 (CBF 0.053, ALS 0.000)
[ĐÚNG] Chọn theo log (NDCG) và theo sở thích thật (Oracle) cho kết quả KHÁC nhau (log thiên lệch): NDCG chọn '(A) tuyến tính, λ = 0.2', Oracle chọn '(B) z-score, w_CBF = 0.5'
[ĐÚNG] CF min-max kém hơn CF thứ hạng phần trăm (công thức A): 0.092 so với 0.138

THAM KHẢO (không phải tiêu chí đạt/không đạt):
  • Theo log, Hybrid thấp hơn CBF: NDCG@10 0.040 so với 0.054. Log thiên vị hệ thống đã hiển thị tin (hệ dựa trên nội dung), xem notebook 06 mục 8.
  • Fold-in đạt 84% Oracle-P@10 của huấn luyện lại đầy đủ (0.151 so với 0.181). Ở quy mô hiện tại huấn luyện lại ALS chỉ mất khoảng 1 giây, nên ứng dụng huấn luyện lại khi có tương tác mới; fold-in là phương án khi dữ liệu lớn.

Thông số cho src/recommender/hybrid.py:
 

## 7. Kết luận

1. **Chọn công thức theo sở thích thật, không theo log.** Nếu chọn theo NDCG của log, công thức trộn tuyến tính (A) thắng; nếu chọn theo Oracle-P@10, công thức z-score (B) thắng (mục 3, và ô kiểm tra ở mục 6). Vì log thiên vị hệ thống đã hiển thị tin (notebook 06), ta chọn theo Oracle. Đây là bài học quan trọng khi đánh giá hệ gợi ý trên log.
2. **Công thức dùng trong ứng dụng: z-score**, $w \cdot z(	ext{CBF}) + (1 - w) \cdot z(	ext{ALS})$ với $w$ ở ô kiểm tra mục 6; tin mới (ALS không biết) chỉ dùng $z(	ext{CBF})$. Công thức này vừa tăng mạnh Oracle-P@10 so với CBF, vừa **không làm chìm tin mới** (Recall@10 trên tin mới không kém CBF).
3. **Trộn tuyến tính cần chuẩn hóa đúng:** CF min-max kém hẳn CF theo thứ hạng phần trăm vì điểm ALS lệch mạnh; nhưng kể cả chuẩn hóa đúng, trộn tuyến tính phải đánh đổi giữa độ chính xác và tin mới (λ lớn ➔ tin mới bị chìm).
4. **Ngưỡng bật CF** lấy từ mục 4 (theo Oracle-P@10): dưới ngưỡng, ứng dụng dùng CBF thuần.
5. **Người dùng mới:** fold-in vẫn hơn CBF nhưng kém huấn luyện lại đầy đủ; ở quy mô hiện tại ứng dụng huấn luyện lại ALS khi log tương tác thay đổi (khoảng 1 giây).
6. **CF học được khẩu vị ẩn** (mục 5): top-10 của Hybrid nghiêng về loại công ty mà người dùng thực sự thích, điều CBF không thể biết vì hồ sơ không ghi.

**Hiển thị trong ứng dụng:** điểm z-score không có thang 0-100% tự nhiên, nên ứng dụng giữ "Tổng điểm (%)" là điểm CBF (độ phù hợp nội dung, như notebook 05), thêm cột "Điểm CF (%)" (thứ hạng phần trăm theo hành vi của người giống bạn), và **xếp hạng theo Hybrid**.

**Giới hạn:** toàn bộ dựa trên dữ liệu mô phỏng. Khi ứng dụng có log thật (`data/interactions/`), cần chạy lại các phép đo này trên log thật, có hiệu chỉnh thiên lệch hiển thị.